# DT-CWT polar debug (freq-only)

Instrumentação da linha experimental de **apenas frequência** (`disable_flow` / `disable_photo`).

Objetivos:
1. Documentar shapes `Yl` / `Yh` do pytorch-wavelets (`[B,C,6,H,W,2]`).
2. Comparar `abs`/`angle` no tensor real (legado) vs polar complexo (`view_as_complex`).
3. Visualizar before/after com parâmetros forçados nos modos `legacy` e `complex`.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.transformer import LightweightDeIdentifier

DEVICE = torch.device("cpu")
IMAGE_SIZE = 224
torch.manual_seed(0)


## Shapes da DT-CWT (J=3)


In [ ]:
m = LightweightDeIdentifier(
    transform_type="dtcwt",
    wavelet_J=3,
    wavelet_polar_mode="complex",
    disable_flow=True,
    disable_photo=True,
).to(DEVICE)

x = torch.rand(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
Yl, Yh = m.wavelet(x)
print("Yl", tuple(Yl.shape), Yl.dtype)
for i, yh in enumerate(Yh):
    print(f"Yh[{i}]", tuple(yh.shape), yh.dtype, "complex?", torch.is_complex(yh))


## Legado vs complexo no mesmo coeficiente


In [ ]:
coeffs = Yh[0]  # [B,C,6,H,W,2]
print("sample real/imag", coeffs[0, 0, 0, 0, 0].detach().cpu().numpy())

mag_legacy = torch.abs(coeffs)
phase_legacy = torch.angle(coeffs)
print("legacy mag/phase shapes", tuple(mag_legacy.shape), tuple(phase_legacy.shape))
print("legacy sample mag/phase", mag_legacy[0,0,0,0,0].tolist(), phase_legacy[0,0,0,0,0].tolist())

c = torch.view_as_complex(coeffs.contiguous())
mag_c = torch.abs(c)
phase_c = torch.angle(c)
print("complex mag/phase shapes", tuple(mag_c.shape), tuple(phase_c.shape))
print("complex sample |c|, angle", float(mag_c[0,0,0,0,0]), float(phase_c[0,0,0,0,0]))


## Before / after (params forçados, freq-only)


In [ ]:
def force_params(model, mag=2.0, phase=1.0):
    with torch.no_grad():
        model.raw_wavelet_mag_scale.fill_(mag)
        model.raw_wavelet_phase_shift.fill_(phase)

def to_img(t):
    return t[0].detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()

m_legacy = LightweightDeIdentifier(
    transform_type="dtcwt", wavelet_polar_mode="legacy",
    disable_flow=True, disable_photo=True,
).to(DEVICE)
m_complex = LightweightDeIdentifier(
    transform_type="dtcwt", wavelet_polar_mode="complex",
    disable_flow=True, disable_photo=True,
).to(DEVICE)
force_params(m_legacy)
force_params(m_complex)

with torch.no_grad():
    y_leg = m_legacy(x)
    y_cx = m_complex(x)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(to_img(x)); axes[0].set_title("input")
axes[1].imshow(to_img(y_leg)); axes[1].set_title("legacy polar")
axes[2].imshow(to_img(y_cx)); axes[2].set_title("complex polar")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("mean |delta| legacy", float((y_leg - x).abs().mean()))
print("mean |delta| complex", float((y_cx - x).abs().mean()))
print("mean |legacy-complex|", float((y_leg - y_cx).abs().mean()))


## Heatmap dos 36 pesos (opcional, após treino freq-only)


In [ ]:
from pathlib import Path
CKPT_ROOT = Path("/mnt/study-data/dcarvalho/tests/wavelet_polar_ab")

def plot_weight_heatmaps(ckpt_path: Path):
    ck = torch.load(ckpt_path, map_location="cpu")
    sd = ck["state_dict"]
    mag = sd["raw_wavelet_mag_scale"].squeeze(0).detach()
    phase = sd["raw_wavelet_phase_shift"].squeeze(0).detach()
    if mag.dim() != 2:
        print("skip heatmap: unexpected shape", tuple(mag.shape))
        return
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    im0 = axes[0].imshow(mag.numpy(), aspect="auto")
    axes[0].set_title("raw mag (orient x level)")
    fig.colorbar(im0, ax=axes[0])
    im1 = axes[1].imshow(phase.numpy(), aspect="auto")
    axes[1].set_title("raw phase (orient x level)")
    fig.colorbar(im1, ax=axes[1])
    plt.tight_layout()
    plt.show()

# Descomente após A/B:
# plot_weight_heatmaps(CKPT_ROOT / "freq_complex/transform.pt")
print("Pronto. Descomente plot_weight_heatmaps quando o A/B terminar.")
